# Step 4 — QC: PCA NAT vs GTEx (batch check)

Auto-converted from `scripts/04_qc_pca_nat_vs_gtex.py` — code preserved verbatim; section banners lifted into headings; paths made portable (`REPO_ROOT`).

In [ ]:
# --- portable repo-root resolution (added during .py -> .ipynb conversion) ---
from pathlib import Path as _Path
def _find_repo_root():
    for _p in [_Path.cwd(), *_Path.cwd().parents]:
        if (_p / "scripts").is_dir() and (_p / "README.md").is_file():
            return _p
    return _Path.cwd()
REPO_ROOT = _find_repo_root()
print("REPO_ROOT =", REPO_ROOT)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import os

### 1. 경로 설정

In [ ]:
BASE_DIR = str(REPO_ROOT)
EXPR_FILE = os.path.join(BASE_DIR, "data_processed/expression/coad_expression_tpm_genes_by_samples.tsv")
META_FILE = os.path.join(BASE_DIR, "data_processed/metadata/coad_sample_labels.tsv")
OUT_DIR   = os.path.join(BASE_DIR, "results/qc")
os.makedirs(OUT_DIR, exist_ok=True)

### 2. 데이터 로드

In [ ]:
print("Loading expression matrix...")
df = pd.read_csv(EXPR_FILE, sep="\t", index_col=0)  # 행=유전자, 열=샘플

print("Loading sample labels...")
meta = pd.read_csv(META_FILE, sep="\t", index_col=0)

### 3. NAT + GTEx만 필터링 (Tumor 제외)

In [ ]:
keep_groups = ["TCGA_COAD_NAT", "GTEx_Colon_Transverse", "GTEx_Colon_Sigmoid"]
meta_sub = meta[meta["group"].isin(keep_groups)]

### 4. 해당 샘플만 expression에서 추출 후 전치 (행=샘플, 열=유전자)

In [ ]:
common_samples = meta_sub.index.intersection(df.columns)
X = df[common_samples].T

print(f"샘플 수: {X.shape[0]}, 유전자 수: {X.shape[1]}")
print(meta_sub["group"].value_counts())

### 5. log2 변환 (이미 돼있으면 스킵)

In [ ]:
if X.values.max() > 50:
    print("log2(TPM+1) 변환 적용...")
    X = np.log2(X + 1)

### 6. 분산 상위 3000 유전자만 사용

In [ ]:
gene_var = X.var(axis=0)
top_genes = gene_var.nlargest(3000).index
X = X[top_genes]

### 7. 스케일링 + PCA

In [ ]:
print("PCA 실행 중...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

### 8. 시각화

In [ ]:
colors = {
    "TCGA_COAD_NAT":       "red",
    "GTEx_Colon_Transverse": "blue",
    "GTEx_Colon_Sigmoid":    "green"
}

plt.figure(figsize=(8, 6))
for group in keep_groups:
    idx = meta_sub.loc[common_samples, "group"] == group
    plt.scatter(
        pca_result[idx.values, 0],
        pca_result[idx.values, 1],
        c=colors[group],
        label=group,
        alpha=0.6,
        s=40
    )

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("PCA: NAT vs GTEx (Tumor 제외)")
plt.legend()
plt.tight_layout()

out_path = os.path.join(OUT_DIR, "pca_nat_vs_gtex.png")
plt.savefig(out_path, dpi=150)
plt.show()
print(f"저장 완료: {out_path}")